# Model BIS — EEG 1D CNN + Numeric MLP → Dual-Head BIS Regressor
**Multimodal Intraoperative Prediction System — BIS Modality**

Predicts the Bispectral Index (BIS) at five time points from EEG signals:
- **Branch A (waveform)**: 1-D CNN on raw EEG (both channels, 128 Hz)
- **Branch B (numerics)**: 2-layer MLP on SEF + SR scalars
- **Fusion**: Concatenation → Linear → 64-dim shared embedding (passed to multimodal stack)
- **Heads**: Independent linear heads for BIS at t (current), t+30 s, t+60 s, t+180 s, and t+300 s
- **NaN handling**: Interpolation → forward-fill → backward-fill for scalar gaps; EEG NaNs zeroed
- **EEG sample rate**: 128 Hz

In [1]:
# ─────────────────────────────────────────────────────────────────────────────
# 1. Imports and seed setting
# ─────────────────────────────────────────────────────────────────────────────
from __future__ import annotations
import gc
import logging
import math
import os
import random
import time
from typing import Dict, List, Optional, Tuple

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from torch.amp import GradScaler, autocast
from torch.utils.data import DataLoader, Dataset
from tqdm import tqdm
import vitaldb

# Reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

In [2]:
# ─────────────────────────────────────────────────────────────────────────────
# 2. CONFIG dict — ALL hyperparameters, track names, file paths
# ─────────────────────────────────────────────────────────────────────────────
CONFIG: Dict = {
    # ── Identity ─────────────────────────────────────────────────────────────
    "model_name":  "model_bis",
    "log_file":    "model_bis.log",

    # ── VitalDB tracks ────────────────────────────────────────────────────────
    # Guaranteed tracks — always present
    "wav_tracks":    ["BIS/EEG1_WAV", "BIS/EEG2_WAV"],  # 128 Hz raw EEG (both channels)
    "wav_interval":  1 / 128,                              # seconds per sample

    # Target + optional scalar subparameters
    "scalar_tracks": [
        "BIS/BIS",   # prediction target (1 Hz, 0–100)
        "BIS/SEF",   # Spectral Edge Frequency — always try, may be missing
        "BIS/SR",    # Suppression Ratio       — always try, may be missing
    ],
    "scalar_interval": 1,   # 1 Hz

    # Column indices within scalar array (order matches scalar_tracks above)
    "bis_col":  0,
    "sef_col":  1,
    "sr_col":   2,

    # ── Case eligibility ──────────────────────────────────────────────────────
    "min_case_duration_sec": 1800,   # 30-minute minimum

    # ── NaN handling for scalar channels ─────────────────────────────────────
    "max_fill_gap_sec":  60,   # interpolate / fill gaps up to 60 s
    "max_nan_frac":      0.30, # discard window if > 30 % NaN in BIS column

    # ── EEG preprocessing ─────────────────────────────────────────────────────
    "eeg_fs":             128,    # Hz
    "eeg_window_sec":     30,     # seconds of EEG per prediction step
    "eeg_samples":        3840,   # 30 × 128
    "eeg_clip_uv":        500.0,  # clip raw EEG to ±500 µV

    # ── Window / stride ───────────────────────────────────────────────────────
    "window_sec":  300,   # 5-minute context window at 1 Hz → 300 scalar steps (unchanged)
    "stride_sec":  60,    # 1-minute stride

    # ── Prediction horizons ───────────────────────────────────────────────────
    "horizon_t0_sec":    0,   # t      — current BIS value
    "horizon_t30_sec":  30,   # t+30 s
    "horizon_t60_sec":  60,   # t+60 s
    "horizon_t180_sec": 180,  # t+180 s
    "horizon_t300_sec": 300,  # t+300 s

    # ── CNN branch ────────────────────────────────────────────────────────────
    "cnn_channels":   [2, 16, 32, 32],   # input=2 EEG channels, 3 conv layers
    "cnn_kernels":    [64, 16, 8],        # kernel sizes per layer
    "cnn_strides":    [4, 2, 2],          # strides per layer
    "cnn_out_dim":    32,                 # after global avg pool

    # ── MLP branch ────────────────────────────────────────────────────────────
    # Input = 2 scalars (SEF, SR) + 2 missing flags = 4
    "mlp_in_dim":    6,  # last (2) + mean (2) + flags (2)
    "mlp_hidden":    32,
    "mlp_out_dim":   32,

    # ── Fusion ────────────────────────────────────────────────────────────────
    "fusion_in_dim":  64,   # cnn_out_dim + mlp_out_dim
    "embedding_dim":  64,   # shared embedding passed to multimodal stack

    # ── Training ──────────────────────────────────────────────────────────────
    "epochs":                    50,
    "batch_size":                16,
    "lr":                        1e-3,
    "weight_decay":              1e-4,
    "gradient_accumulation_steps": 2,
    "early_stopping_patience":   8,
    "max_grad_norm":             1.0,

    # ── Data split ────────────────────────────────────────────────────────────
    "val_frac":   0.10,
    "test_frac":  0.10,
    "seed":       42,

    # ── Output files ──────────────────────────────────────────────────────────
    "checkpoint":    "ckpt_model_bis.pt",
    "pred_output":   "prediction_model_bis.png",
}

In [3]:
# ─────────────────────────────────────────────────────────────────────────────
# 3. Logging setup
# ─────────────────────────────────────────────────────────────────────────────
def setup_logging(log_file: str) -> logging.Logger:
    """Configure root logger with file and console handlers."""
    fmt = logging.Formatter(
        "%(asctime)s | %(levelname)-8s | %(name)s | %(message)s",
        datefmt="%Y-%m-%d %H:%M:%S",
    )
    logger = logging.getLogger("model_bis")
    logger.setLevel(logging.DEBUG)

    ch = logging.StreamHandler()
    ch.setLevel(logging.INFO)
    ch.setFormatter(fmt)
    logger.addHandler(ch)

    fh = logging.FileHandler(log_file, mode="w")
    fh.setLevel(logging.DEBUG)
    fh.setFormatter(fmt)
    logger.addHandler(fh)

    return logger

log = setup_logging(CONFIG["log_file"])

In [4]:
# ─────────────────────────────────────────────────────────────────────────────
# 4. Data loading — EEG waveforms (128 Hz) + BIS scalars (1 Hz)
# ─────────────────────────────────────────────────────────────────────────────
def load_data(
    caseids: List[int],
    config: Dict,
) -> Tuple[Dict[int, np.ndarray], Dict[int, np.ndarray]]:
    """Load raw EEG waveforms and BIS scalar tracks for each case.

    Parameters
    ----------
    caseids : List[int]
    config  : Dict

    Returns
    -------
    wav_data    : Dict[int, np.ndarray]  caseid → float32 (T_wav, 2) at 128 Hz
    scalar_data : Dict[int, np.ndarray]  caseid → float32 (T_1hz, 3) at 1 Hz
                  columns: [BIS, SEF, SR]
    """
    wav_data:    Dict[int, np.ndarray] = {}
    scalar_data: Dict[int, np.ndarray] = {}

    for caseid in tqdm(caseids, desc="Loading cases"):
        try:
            arr_wav = vitaldb.load_case(
                caseid=caseid,
                track_names=config["wav_tracks"],
                interval=config["wav_interval"],
            )
        except Exception as exc:
            log.warning(f"caseid {caseid}: wav load failed — {exc}")
            continue

        if arr_wav is None:
            continue

        try:
            arr_scalar = vitaldb.load_case(
                caseid=caseid,
                track_names=config["scalar_tracks"],
                interval=config["scalar_interval"],
            )
        except Exception as exc:
            log.warning(f"caseid {caseid}: scalar load failed — {exc}")
            continue

        if arr_scalar is None:
            continue

        # BIS/BIS must exist to be a valid case
        bis_col = arr_scalar[:, config["bis_col"]]
        if np.all(np.isnan(bis_col)):
            log.debug(f"caseid {caseid}: BIS all-NaN, skipping")
            continue

        if arr_scalar.shape[0] < config["min_case_duration_sec"]:
            log.debug(f"caseid {caseid}: too short, skipping")
            continue

        wav_data[caseid]    = arr_wav.astype(np.float32)
        scalar_data[caseid] = arr_scalar.astype(np.float32)

    log.info(
        f"Loaded {len(wav_data)} cases "        f"({len(caseids) - len(wav_data)} skipped)"
    )
    return wav_data, scalar_data

In [5]:
# ─────────────────────────────────────────────────────────────────────────────
# 5. NaN handling — interpolate → forward-fill → backward-fill
#    Applied column-wise to scalar arrays.
#    Gaps longer than max_gap_sec are left as NaN for later window discard.
# ─────────────────────────────────────────────────────────────────────────────
def fill_nan_gaps(arr: np.ndarray, max_gap_sec: int) -> np.ndarray:
    """Fill NaN runs up to max_gap_sec using linear interpolation where
    both boundary neighbours exist, forward-fill for trailing gaps,
    backward-fill for leading gaps.

    Parameters
    ----------
    arr         : np.ndarray  shape (T, F)  — operated column-wise
    max_gap_sec : int         maximum consecutive NaNs to fill

    Returns
    -------
    np.ndarray  same shape, NaN gaps up to max_gap_sec filled
    """
    out = arr.copy()
    T, F = out.shape

    for f in range(F):
        col = out[:, f].copy()
        nan_mask = np.isnan(col)
        if not nan_mask.any():
            continue

        # Identify contiguous NaN runs
        nan_idx = np.where(nan_mask)[0]
        gaps = np.split(nan_idx, np.where(np.diff(nan_idx) != 1)[0] + 1)

        for gap in gaps:
            gap_len = len(gap)
            if gap_len > max_gap_sec:
                continue   # too long — leave as NaN

            start, end = gap[0], gap[-1]
            has_left   = start > 0 and not np.isnan(col[start - 1])
            has_right  = end < T - 1 and not np.isnan(col[end + 1])

            if has_left and has_right:
                # Linear interpolation between known boundary values
                col[gap] = np.linspace(col[start - 1], col[end + 1], gap_len + 2)[1:-1]
            elif has_left:
                col[gap] = col[start - 1]   # forward-fill
            elif has_right:
                col[gap] = col[end + 1]     # backward-fill
            # else: isolated island with no valid neighbours → leave as NaN

        out[:, f] = col

    return out


def preprocess_scalars(arr: np.ndarray, config: Dict) -> np.ndarray:
    """Clip physiological outliers to NaN, then apply gap filling.

    Physiological bounds
    --------------------
    BIS : 0–100
    SEF : 0–50  Hz
    SR  : 0–100 %

    Parameters
    ----------
    arr    : np.ndarray  (T, 3) columns [BIS, SEF, SR]
    config : Dict

    Returns
    -------
    np.ndarray  same shape, clipped and gap-filled
    """
    bounds = [(0.0, 100.0), (0.0, 50.0), (0.0, 100.0)]
    out = arr.copy()
    for col_idx, (lo, hi) in enumerate(bounds):
        col = out[:, col_idx]
        out[:, col_idx] = np.where((col < lo) | (col > hi), np.nan, col)

    out = fill_nan_gaps(out, max_gap_sec=config["max_fill_gap_sec"])
    return out

In [6]:
# ─────────────────────────────────────────────────────────────────────────────
# 6. Window generation
#    Each window uses a context_sec scalar history and the final 30-s EEG
#    segment. Labels are BIS values at t, t+30 s, t+60 s, t+180 s, and t+300 s.
# ─────────────────────────────────────────────────────────────────────────────
def generate_windows(
    wav_data:    Dict[int, np.ndarray],
    scalar_data: Dict[int, np.ndarray],
    config:      Dict,
) -> List[Dict]:
    """Slide windows over each case and extract model inputs + BIS labels.

    Window contents
    ---------------
    eeg           : np.ndarray (eeg_samples, 2) = (3840, 2)  — last 30-s raw EEG
    sef_sr        : np.ndarray (window_sec, 2)  = (300, 2)   — SEF, SR context
    sef_sr_missing: np.ndarray (2,)  — binary flags: 1.0 if column all-NaN
    bis_t0        : float  — BIS at t       (current value)
    bis_t30       : float  — BIS at t+30 s
    bis_t60       : float  — BIS at t+60 s
    bis_t180      : float  — BIS at t+180 s
    bis_t300      : float  — BIS at t+300 s

    Parameters
    ----------
    wav_data    : Dict[int, np.ndarray]
    scalar_data : Dict[int, np.ndarray]
    config      : Dict

    Returns
    -------
    List[Dict]
    """
    windows: List[Dict] = []

    win_sec    = config["window_sec"]        # 300
    stride_sec = config["stride_sec"]        # 60
    h0         = config["horizon_t0_sec"]    # 0   — current
    h30        = config["horizon_t30_sec"]   # 30
    h60        = config["horizon_t60_sec"]   # 60
    h180       = config["horizon_t180_sec"]  # 180
    h300       = config["horizon_t300_sec"]  # 300
    eeg_n      = config["eeg_samples"]       # 3840
    eeg_fs     = config["eeg_fs"]            # 128
    max_nan    = config["max_nan_frac"]      # 0.30
    bis_col    = config["bis_col"]
    sef_col    = config["sef_col"]
    sr_col     = config["sr_col"]
    clip_uv    = config["eeg_clip_uv"]

    for caseid, arr_scalar in tqdm(scalar_data.items(), desc="Generating windows"):
        arr_wav = wav_data.get(caseid)
        if arr_wav is None:
            continue

        T_scalar = arr_scalar.shape[0]
        need_end = win_sec + h300  # latest window end we'll need labels for

        if T_scalar < need_end:
            log.debug(f"caseid {caseid}: too short for any window+horizon")
            continue

        # Pre-process scalar channels
        arr_scalar = preprocess_scalars(arr_scalar, config)

        # Detect globally missing optional columns (all-NaN in this case)
        sef_missing = float(np.all(np.isnan(arr_scalar[:, sef_col])))
        sr_missing  = float(np.all(np.isnan(arr_scalar[:, sr_col])))
        missing_flags = np.array([sef_missing, sr_missing], dtype=np.float32)

        bis_series  = arr_scalar[:, bis_col]
        sef_series  = arr_scalar[:, sef_col]
        sr_series   = arr_scalar[:, sr_col]

        for t_end in range(win_sec, T_scalar - h300, stride_sec):
            t_start = t_end - win_sec

            # BIS window quality gate — discard if too many NaNs
            bis_win = bis_series[t_start:t_end]
            if np.isnan(bis_win).mean() > max_nan:
                continue

            # Targets — skip window if any label is NaN
            # t0: current value (last step of the context window)
            bis_t0   = bis_series[t_end - 1]                                           if t_end         <= T_scalar else np.nan
            bis_t30  = bis_series[t_end + h30  - 1] if h30  > 0 and (t_end + h30)  <= T_scalar else np.nan
            bis_t60  = bis_series[t_end + h60  - 1] if h60  > 0 and (t_end + h60)  <= T_scalar else np.nan
            bis_t180 = bis_series[t_end + h180 - 1] if h180 > 0 and (t_end + h180) <= T_scalar else np.nan
            bis_t300 = bis_series[t_end + h300 - 1] if h300 > 0 and (t_end + h300) <= T_scalar else np.nan
            if any(np.isnan(v) for v in [bis_t0, bis_t30, bis_t60, bis_t180, bis_t300]):
                continue

            # SEF + SR context window (300, 2)
            sef_win = sef_series[t_start:t_end]
            sr_win  = sr_series[t_start:t_end]
            sef_sr  = np.stack([sef_win, sr_win], axis=-1)   # (300, 2)
            # Replace residual NaN with 0 (missing_flags already capture this)
            sef_sr  = np.nan_to_num(sef_sr, nan=0.0)

            # EEG window — last 30 s before t_end
            wav_end   = int(t_end * eeg_fs)
            wav_start = max(0, wav_end - eeg_n)
            if wav_end > arr_wav.shape[0]:
                continue
            eeg = arr_wav[wav_start:wav_end, :]   # (3840, 2)
            if eeg.shape[0] < eeg_n:
                continue
            eeg = np.nan_to_num(eeg, nan=0.0)
            eeg = np.clip(eeg, -clip_uv, clip_uv).astype(np.float32)

            windows.append({
                "caseid":        caseid,
                "window_start":  t_start,
                "eeg":           eeg,                        # (3840, 2)
                "sef_sr":        sef_sr.astype(np.float32),  # (300, 2)
                "missing_flags": missing_flags,              # (2,)
                "bis_t0":        np.float32(bis_t0),
                "bis_t30":       np.float32(bis_t30),
                "bis_t60":       np.float32(bis_t60),
                "bis_t180":      np.float32(bis_t180),
                "bis_t300":      np.float32(bis_t300),
            })

        del arr_wav
        if caseid in wav_data:
            del wav_data[caseid]
        gc.collect()

    log.info(f"Total windows generated: {len(windows)}")
    return windows

In [7]:
# ─────────────────────────────────────────────────────────────────────────────
# 7. Scaler fitting and application (training data only)
# ─────────────────────────────────────────────────────────────────────────────
def fit_scalers(
    train_windows: List[Dict],
) -> Tuple[np.ndarray, np.ndarray, np.ndarray, np.ndarray]:
    """Compute per-channel statistics on training windows.

    Returns
    -------
    eeg_mean   : (2,)   per-channel EEG mean
    eeg_std    : (2,)   per-channel EEG std (clipped to ≥1e-6)
    sef_sr_mean: (2,)   per-column SEF/SR mean
    sef_sr_std : (2,)   per-column SEF/SR std
    """
    all_eeg    = np.concatenate([w["eeg"].reshape(-1, 2) for w in train_windows], axis=0)
    all_sef_sr = np.concatenate([w["sef_sr"].reshape(-1, 2) for w in train_windows], axis=0)

    eeg_mean    = all_eeg.mean(axis=0)
    eeg_std     = all_eeg.std(axis=0).clip(min=1e-6)
    sef_sr_mean = np.nanmean(all_sef_sr, axis=0)
    sef_sr_std  = np.nanstd(all_sef_sr,  axis=0).clip(min=1e-6)

    log.info(f"EEG mean={eeg_mean}, std={eeg_std}")
    log.info(f"SEF/SR mean={sef_sr_mean}, std={sef_sr_std}")
    return eeg_mean, eeg_std, sef_sr_mean, sef_sr_std


def apply_scalers(
    windows:     List[Dict],
    eeg_mean:    np.ndarray,
    eeg_std:     np.ndarray,
    sef_sr_mean: np.ndarray,
    sef_sr_std:  np.ndarray,
) -> List[Dict]:
    """Normalise EEG and SEF/SR in-place using training statistics."""
    for w in windows:
        w["eeg"]    = ((w["eeg"]    - eeg_mean)    / eeg_std).astype(np.float32)
        w["sef_sr"] = ((w["sef_sr"] - sef_sr_mean) / sef_sr_std).astype(np.float32)
    return windows

In [8]:
# ─────────────────────────────────────────────────────────────────────────────
# 8. Train / Val / Test split (patient-level, no leakage)
# ─────────────────────────────────────────────────────────────────────────────
def split_cases(
    all_caseids: List[int],
    val_frac:    float = 0.10,
    test_frac:   float = 0.10,
    seed:        int   = 42,
) -> Tuple[List[int], List[int], List[int]]:
    """Patient-level split — a case never appears in more than one set."""
    train_val, test = train_test_split(all_caseids, test_size=test_frac, random_state=seed)
    val_size        = val_frac / (1.0 - test_frac)
    train, val      = train_test_split(train_val, test_size=val_size, random_state=seed)
    log.info(f"Split: train={len(train)}, val={len(val)}, test={len(test)} cases")
    return train, val, test

In [9]:
# ─────────────────────────────────────────────────────────────────────────────
# 9. BISDataset — PyTorch Dataset
# ─────────────────────────────────────────────────────────────────────────────
class BISDataset(Dataset):
    """PyTorch Dataset for BIS regression windows.

    Parameters
    ----------
    windows : List[Dict]  pre-processed window dicts
    """

    def __init__(self, windows: List[Dict]) -> None:
        self.windows = windows

    def __len__(self) -> int:
        return len(self.windows)

    def __getitem__(self, idx: int) -> Dict:
        """
        Returns
        -------
        Dict:
            eeg           : FloatTensor (3840, 2)  — normalised raw EEG
            sef_sr        : FloatTensor (300, 2)   — normalised SEF + SR context
            missing_flags : FloatTensor (2,)       — 1.0 if column all-NaN
            bis_t0        : FloatTensor (1,)       — BIS at t       (current)
            bis_t30       : FloatTensor (1,)       — BIS at t+30 s
            bis_t60       : FloatTensor (1,)       — BIS at t+60 s
            bis_t180      : FloatTensor (1,)       — BIS at t+180 s
            bis_t300      : FloatTensor (1,)       — BIS at t+300 s
        """
        w = self.windows[idx]
        return {
            "caseid":        torch.tensor(w["caseid"]),
            "eeg":           torch.FloatTensor(w["eeg"]),
            "sef_sr":        torch.FloatTensor(w["sef_sr"]),
            "missing_flags": torch.FloatTensor(w["missing_flags"]),
            "bis_t0":        torch.FloatTensor([w["bis_t0"]]),
            "bis_t30":       torch.FloatTensor([w["bis_t30"]]),
            "bis_t60":       torch.FloatTensor([w["bis_t60"]]),
            "bis_t180":      torch.FloatTensor([w["bis_t180"]]),
            "bis_t300":      torch.FloatTensor([w["bis_t300"]]),
        }

In [10]:
# ─────────────────────────────────────────────────────────────────────────────
# 10. DataLoaders
# ─────────────────────────────────────────────────────────────────────────────
def build_dataloaders(
    train_ds: BISDataset,
    val_ds:   BISDataset,
    test_ds:  BISDataset,
    config:   Dict,
) -> Tuple[DataLoader, DataLoader, DataLoader]:
    """Build train / val / test DataLoaders."""
    bs = config["batch_size"]
    train_loader = DataLoader(train_ds, batch_size=bs, shuffle=True,
                              num_workers=2, pin_memory=True, drop_last=True)
    val_loader   = DataLoader(val_ds,   batch_size=bs, shuffle=False,
                              num_workers=2, pin_memory=True)
    test_loader  = DataLoader(test_ds,  batch_size=bs, shuffle=False,
                              num_workers=2, pin_memory=True)
    log.info(
        f"DataLoaders: train={len(train_loader)} batches, "
        f"val={len(val_loader)}, test={len(test_loader)}"
    )
    return train_loader, val_loader, test_loader

In [11]:
# ─────────────────────────────────────────────────────────────────────────────
# 11. Model architecture
#
#   Branch A — CNN on raw EEG (3840, 2)
#     3 × (Conv1d → BatchNorm → ReLU) with strided downsampling
#     → Global Average Pooling → dim 32
#
#   Branch B — MLP on SEF + SR + missing flags (4 inputs)
#     Linear(4→32) → ReLU → Linear(32→32) → ReLU
#     The (300, 2) time-series is mean-pooled to (2,) before concatenating
#     with the 2 missing-flag scalars → 4-dim input
#
#   Fusion — Concat(32+32=64) → Linear(64→64) → ReLU → embedding (64-dim)
#
#   Heads  — Linear(64→1) each for BIS at t, t+30, t+60, t+180, t+300
# ─────────────────────────────────────────────────────────────────────────────

class CNNBranch(nn.Module):
    """1-D CNN encoder for raw dual-channel EEG.

    Input : (B, 2, 3840)
    Output: (B, cnn_out_dim)  — after global average pool
    """

    def __init__(self, config: Dict) -> None:
        super().__init__()
        channels = config["cnn_channels"]   # [2, 16, 32, 32]
        kernels  = config["cnn_kernels"]    # [64, 16, 8]
        strides  = config["cnn_strides"]    # [4, 2, 2]

        layers: List[nn.Module] = []
        for i in range(len(kernels)):
            layers += [
                nn.Conv1d(channels[i], channels[i + 1],
                          kernel_size=kernels[i], stride=strides[i],
                          padding=kernels[i] // 2, bias=False),
                nn.BatchNorm1d(channels[i + 1]),
                nn.ReLU(inplace=True),
            ]
        self.conv_stack = nn.Sequential(*layers)
        self.pool       = nn.AdaptiveAvgPool1d(1)   # global avg pool → (B, C, 1)
        self.out_dim    = config["cnn_out_dim"]      # 32

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """
        Parameters
        ----------
        x : (B, 2, 3840)

        Returns
        -------
        (B, cnn_out_dim)
        """
        h = self.conv_stack(x)   # (B, 32, T')
        h = self.pool(h)         # (B, 32, 1)
        return h.squeeze(-1)     # (B, 32)


class MLPBranch(nn.Module):
    """2-layer MLP on mean-pooled SEF/SR + missing-indicator flags.

    Input : sef_sr (B, 300, 2) + missing_flags (B, 2)
    Output: (B, mlp_out_dim)
    """

    def __init__(self, config: Dict) -> None:
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(config["mlp_in_dim"], config["mlp_hidden"]),
            nn.ReLU(inplace=True),
            nn.Linear(config["mlp_hidden"], config["mlp_out_dim"]),
            nn.ReLU(inplace=True),
        )

    def forward(
        self,
        sef_sr:        torch.Tensor,
        missing_flags: torch.Tensor,
    ) -> torch.Tensor:
        """
        Parameters
        ----------
        sef_sr        : (B, 300, 2)
        missing_flags : (B, 2)

        Returns
        -------
        (B, mlp_out_dim)
        """
        # Last-value + mean concat — preserves trend signal vs pure mean
        last   = sef_sr[:, -1, :]                             # (B, 2)  last timestep
        mean   = sef_sr.mean(dim=1)                           # (B, 2)  global mean
        x      = torch.cat([last, mean, missing_flags], dim=-1)  # (B, 6)
        return self.net(x)


class BISModel(nn.Module):
    """Full BIS prediction model: CNN + MLP → fusion embedding → dual heads.

    Attributes
    ----------
    cnn_branch : CNNBranch
    mlp_branch : MLPBranch
    fusion     : nn.Sequential   Linear → ReLU  (64 → 64)
    head_t0    : nn.Linear       predicts BIS at t (current)
    head_t30   : nn.Linear       predicts BIS at t+30 s
    head_t60   : nn.Linear       predicts BIS at t+60 s
    head_t180  : nn.Linear       predicts BIS at t+180 s
    head_t300  : nn.Linear       predicts BIS at t+300 s

    The 64-dim output of ``fusion`` is the embedding passed to the
    multimodal fusion stack.
    """

    def __init__(self, config: Dict) -> None:
        super().__init__()
        self.cnn_branch = CNNBranch(config)
        self.mlp_branch = MLPBranch(config)

        self.fusion = nn.Sequential(
            nn.Linear(config["fusion_in_dim"], config["embedding_dim"]),
            nn.ReLU(inplace=True),
        )

        emb_dim       = config["embedding_dim"]
        self.head_t0   = nn.Linear(emb_dim, 1)
        self.head_t30  = nn.Linear(emb_dim, 1)
        self.head_t60  = nn.Linear(emb_dim, 1)
        self.head_t180 = nn.Linear(emb_dim, 1)
        self.head_t300 = nn.Linear(emb_dim, 1)

    def encode(
        self,
        eeg:           torch.Tensor,
        sef_sr:        torch.Tensor,
        missing_flags: torch.Tensor,
    ) -> torch.Tensor:
        """Produce the 64-dim fusion embedding (used by multimodal stack).

        Parameters
        ----------
        eeg           : (B, 2, 3840)
        sef_sr        : (B, 300, 2)
        missing_flags : (B, 2)

        Returns
        -------
        embedding : (B, 64)
        """
        cnn_feat = self.cnn_branch(eeg)                          # (B, 32)
        mlp_feat = self.mlp_branch(sef_sr, missing_flags)        # (B, 32)
        concat   = torch.cat([cnn_feat, mlp_feat], dim=-1)       # (B, 64)
        return self.fusion(concat)                                # (B, 64)

    def forward(
        self,
        eeg:           torch.Tensor,
        sef_sr:        torch.Tensor,
        missing_flags: torch.Tensor,
    ) -> Dict[str, torch.Tensor]:
        """Full forward pass.

        Returns
        -------
        Dict:
            embedding : (B, 64)   — fusion embedding for multimodal stack
            bis_t0    : (B, 1)    — predicted BIS at t (current)
            bis_t30   : (B, 1)    — predicted BIS at t+30 s
            bis_t60   : (B, 1)    — predicted BIS at t+60 s
            bis_t180  : (B, 1)    — predicted BIS at t+180 s
            bis_t300  : (B, 1)    — predicted BIS at t+300 s
        """
        embedding = self.encode(eeg, sef_sr, missing_flags)
        return {
            "embedding": embedding,
            "bis_t0":    self.head_t0(embedding).clamp(0, 100),
            "bis_t30":   self.head_t30(embedding).clamp(0, 100),
            "bis_t60":   self.head_t60(embedding).clamp(0, 100),
            "bis_t180":  self.head_t180(embedding).clamp(0, 100),
            "bis_t300":  self.head_t300(embedding).clamp(0, 100),
        }

In [12]:
# ─────────────────────────────────────────────────────────────────────────────
# 12. Loss — Huber (smooth L1) summed across both horizons
#     Huber is more robust than plain MSE to occasional outlier BIS values.
# ─────────────────────────────────────────────────────────────────────────────
def bis_loss(
    pred_t0:    torch.Tensor,
    pred_t30:   torch.Tensor,
    pred_t60:   torch.Tensor,
    pred_t180:  torch.Tensor,
    pred_t300:  torch.Tensor,
    target_t0:  torch.Tensor,
    target_t30: torch.Tensor,
    target_t60: torch.Tensor,
    target_t180:torch.Tensor,
    target_t300:torch.Tensor,
    delta:      float = 5.0,
) -> torch.Tensor:
    """Huber loss summed over all five prediction horizons.

    Parameters
    ----------
    pred_t0 / pred_t30 / pred_t60 / pred_t180 / pred_t300 : (B, 1)
    target_t0 / target_t30 / target_t60 / target_t180 / target_t300 : (B, 1)
    delta : float   Huber transition point (BIS units, default 5)

    Returns
    -------
    torch.Tensor  scalar
    """
    loss_t0   = F.huber_loss(pred_t0,   target_t0,   delta=delta)
    loss_t30  = F.huber_loss(pred_t30,  target_t30,  delta=delta)
    loss_t60  = F.huber_loss(pred_t60,  target_t60,  delta=delta)
    loss_t180 = F.huber_loss(pred_t180, target_t180, delta=delta)
    loss_t300 = F.huber_loss(pred_t300, target_t300, delta=delta)
    return loss_t0 + loss_t30 + loss_t60 + loss_t180 + loss_t300

In [13]:
# ─────────────────────────────────────────────────────────────────────────────
# 13. Metrics — MAE and RMSE for each horizon
# ─────────────────────────────────────────────────────────────────────────────
def compute_metrics(
    preds_t0:    np.ndarray,
    preds_t30:   np.ndarray,
    preds_t60:   np.ndarray,
    preds_t180:  np.ndarray,
    preds_t300:  np.ndarray,
    targets_t0:  np.ndarray,
    targets_t30: np.ndarray,
    targets_t60: np.ndarray,
    targets_t180:np.ndarray,
    targets_t300:np.ndarray,
) -> Dict[str, float]:
    """Return MAE and RMSE for each prediction horizon."""
    def mae(p, t):  return float(np.mean(np.abs(p - t)))
    def rmse(p, t): return float(np.sqrt(np.mean((p - t) ** 2)))

    return {
        "mae_t0":    mae(preds_t0,   targets_t0),
        "rmse_t0":   rmse(preds_t0,  targets_t0),
        "mae_t30":   mae(preds_t30,  targets_t30),
        "rmse_t30":  rmse(preds_t30, targets_t30),
        "mae_t60":   mae(preds_t60,  targets_t60),
        "rmse_t60":  rmse(preds_t60, targets_t60),
        "mae_t180":  mae(preds_t180, targets_t180),
        "rmse_t180": rmse(preds_t180,targets_t180),
        "mae_t300":  mae(preds_t300, targets_t300),
        "rmse_t300": rmse(preds_t300,targets_t300),
    }

In [14]:
# ─────────────────────────────────────────────────────────────────────────────
# 14a. One training epoch
# ─────────────────────────────────────────────────────────────────────────────
def train_one_epoch(
    model:      BISModel,
    loader:     DataLoader,
    optimizer:  torch.optim.Optimizer,
    amp_scaler: GradScaler,
    config:     Dict,
    device:     torch.device,
) -> float:
    """Run one supervised training epoch.

    Returns
    -------
    float  mean combined loss
    """
    model.train()
    total_loss = 0.0
    n_batches  = 0
    grad_acc   = config["gradient_accumulation_steps"]
    optimizer.zero_grad()

    for step, batch in enumerate(tqdm(loader, desc="Train", leave=False)):
        # EEG: Dataset stores (3840, 2); Conv1d needs (B, 2, 3840)
        eeg           = batch["eeg"].to(device).permute(0, 2, 1)   # (B, 2, 3840)
        sef_sr        = batch["sef_sr"].to(device)                   # (B, 300, 2)
        missing_flags = batch["missing_flags"].to(device)            # (B, 2)
        target_t0     = batch["bis_t0"].to(device)                   # (B, 1)
        target_t30    = batch["bis_t30"].to(device)                  # (B, 1)
        target_t60    = batch["bis_t60"].to(device)                  # (B, 1)
        target_t180   = batch["bis_t180"].to(device)                 # (B, 1)
        target_t300   = batch["bis_t300"].to(device)                 # (B, 1)

        with autocast(device_type=device.type):
            out  = model(eeg, sef_sr, missing_flags)
            loss = bis_loss(
                out["bis_t0"],   out["bis_t30"],  out["bis_t60"],
                out["bis_t180"], out["bis_t300"],
                target_t0, target_t30, target_t60, target_t180, target_t300,
            ) / grad_acc

        amp_scaler.scale(loss).backward()

        if (step + 1) % grad_acc == 0:
            amp_scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), config["max_grad_norm"])
            amp_scaler.step(optimizer)
            amp_scaler.update()
            optimizer.zero_grad()

        loss_val = loss.item() * grad_acc
        if not math.isfinite(loss_val):
            log.warning(f"Step {step}: non-finite loss, skipping")
        else:
            total_loss += loss_val
            n_batches  += 1

    return total_loss / max(n_batches, 1)


# ─────────────────────────────────────────────────────────────────────────────
# 14b. Validation pass
# ─────────────────────────────────────────────────────────────────────────────
def validate(
    model:  BISModel,
    loader: DataLoader,
    config: Dict,
    device: torch.device,
) -> Tuple[float, Dict[str, float]]:
    """Run one validation pass.

    Returns
    -------
    val_loss : float
    metrics  : Dict[str, float]
    """
    model.eval()
    total_loss = 0.0
    n_batches  = 0
    preds_t0, preds_t30, preds_t60, preds_t180, preds_t300 = [], [], [], [], []
    targets_t0, targets_t30, targets_t60, targets_t180, targets_t300 = [], [], [], [], []

    with torch.no_grad():
        for batch in tqdm(loader, desc="Validate", leave=False):
            eeg           = batch["eeg"].to(device).permute(0, 2, 1)
            sef_sr        = batch["sef_sr"].to(device)
            missing_flags = batch["missing_flags"].to(device)
            target_t0     = batch["bis_t0"].to(device)
            target_t30    = batch["bis_t30"].to(device)
            target_t60    = batch["bis_t60"].to(device)
            target_t180   = batch["bis_t180"].to(device)
            target_t300   = batch["bis_t300"].to(device)

            with autocast(device_type=device.type):
                out  = model(eeg, sef_sr, missing_flags)
                loss = bis_loss(
                    out["bis_t0"],   out["bis_t30"],  out["bis_t60"],
                    out["bis_t180"], out["bis_t300"],
                    target_t0, target_t30, target_t60, target_t180, target_t300,
                )

            total_loss += loss.item()
            n_batches  += 1

            preds_t0.append(out["bis_t0"].cpu().numpy())
            preds_t30.append(out["bis_t30"].cpu().numpy())
            preds_t60.append(out["bis_t60"].cpu().numpy())
            preds_t180.append(out["bis_t180"].cpu().numpy())
            preds_t300.append(out["bis_t300"].cpu().numpy())
            targets_t0.append(target_t0.cpu().numpy())
            targets_t30.append(target_t30.cpu().numpy())
            targets_t60.append(target_t60.cpu().numpy())
            targets_t180.append(target_t180.cpu().numpy())
            targets_t300.append(target_t300.cpu().numpy())

    val_loss    = total_loss / max(n_batches, 1)
    metrics     = compute_metrics(
        np.concatenate(preds_t0).flatten(),
        np.concatenate(preds_t30).flatten(),
        np.concatenate(preds_t60).flatten(),
        np.concatenate(preds_t180).flatten(),
        np.concatenate(preds_t300).flatten(),
        np.concatenate(targets_t0).flatten(),
        np.concatenate(targets_t30).flatten(),
        np.concatenate(targets_t60).flatten(),
        np.concatenate(targets_t180).flatten(),
        np.concatenate(targets_t300).flatten(),
    )
    return val_loss, metrics

In [15]:
# ─────────────────────────────────────────────────────────────────────────────
# 15. EarlyStopping
# ─────────────────────────────────────────────────────────────────────────────
class EarlyStopping:
    """Stop training when val loss has not improved for `patience` epochs."""

    def __init__(self, patience: int = 8, checkpoint_path: str = "best.pt") -> None:
        self.patience        = patience
        self.checkpoint_path = checkpoint_path
        self.best_loss       = float("inf")
        self.counter         = 0
        self.should_stop     = False

    def step(self, val_loss: float, model: nn.Module) -> None:
        if val_loss < self.best_loss:
            self.best_loss = val_loss
            self.counter   = 0
            torch.save(model.state_dict(), self.checkpoint_path)
            log.info(f"  ✓ New best val_loss={val_loss:.4f} — checkpoint saved")
        else:
            self.counter += 1
            log.info(f"  EarlyStopping {self.counter}/{self.patience}")
            if self.counter >= self.patience:
                self.should_stop = True
                log.info("  Early stopping triggered.")

In [16]:
# ─────────────────────────────────────────────────────────────────────────────
# 16. train_model — full supervised training loop
# ─────────────────────────────────────────────────────────────────────────────
def train_model(
    model:        BISModel,
    train_loader: DataLoader,
    val_loader:   DataLoader,
    config:       Dict,
    device:       torch.device,
) -> BISModel:
    """Train with AdamW + cosine LR schedule + early stopping.

    Returns
    -------
    BISModel  best model loaded from checkpoint
    """
    model      = model.to(device)
    amp_scaler = GradScaler(device=device.type)
    optimizer  = torch.optim.AdamW(
        model.parameters(),
        lr=config["lr"],
        weight_decay=config["weight_decay"],
    )
    scheduler  = torch.optim.lr_scheduler.CosineAnnealingLR(
        optimizer, T_max=config["epochs"], eta_min=1e-6
    )
    early_stop = EarlyStopping(
        patience=config["early_stopping_patience"],
        checkpoint_path=config["checkpoint"],
    )

    log.info("=== Training BIS Model ===")
    for epoch in range(1, config["epochs"] + 1):
        t0         = time.time()
        train_loss = train_one_epoch(model, train_loader, optimizer, amp_scaler, config, device)
        val_loss, metrics = validate(model, val_loader, config, device)
        scheduler.step()
        elapsed = time.time() - t0

        log.info(
            f"Epoch {epoch}/{config['epochs']}  "
            f"train={train_loss:.4f}  val={val_loss:.4f}  "
            f"mae_t0={metrics['mae_t0']:.2f}  mae_t30={metrics['mae_t30']:.2f}  "
            f"mae_t60={metrics['mae_t60']:.2f}  mae_t180={metrics['mae_t180']:.2f}  "
            f"mae_t300={metrics['mae_t300']:.2f}  ({elapsed:.1f}s)"
        )
        early_stop.step(val_loss, model)
        if early_stop.should_stop:
            break

    model.load_state_dict(
        torch.load(config["checkpoint"], map_location=device, weights_only=True)
    )
    log.info("Best model loaded from checkpoint.")
    return model

In [17]:
# ─────────────────────────────────────────────────────────────────────────────
# 17. Test evaluation
# ─────────────────────────────────────────────────────────────────────────────
def evaluate_test(
    model:       BISModel,
    test_loader: DataLoader,
    config:      Dict,
    device:      torch.device,
) -> Dict[str, float]:
    """Evaluate on held-out test set and log metrics."""
    test_loss, metrics = validate(model, test_loader, config, device)
    log.info("=== Test Set Evaluation ===")
    log.info(f"  Test loss: {test_loss:.4f}")
    for k, v in metrics.items():
        log.info(f"  {k}: {v:.4f}")
    return metrics

In [18]:
# ─────────────────────────────────────────────────────────────────────────────
# 18. Prediction visualisation
# ─────────────────────────────────────────────────────────────────────────────
def plot_predictions(
    model:       BISModel,
    test_loader: DataLoader,
    config:      Dict,
    device:      torch.device,
    n_cases:     int = 4,
) -> None:
    """Plot predicted vs true BIS for t+5 and t+10 min across n_cases.

    Saves to config['pred_output'].
    """
    model.eval()
    all_pred_t0, all_pred_t30, all_pred_t60, all_pred_t180, all_pred_t300 = [], [], [], [], []
    all_true_t0, all_true_t30, all_true_t60, all_true_t180, all_true_t300 = [], [], [], [], []
    all_caseids = []

    with torch.no_grad():
        for batch in test_loader:
            eeg           = batch["eeg"].to(device).permute(0, 2, 1)
            sef_sr        = batch["sef_sr"].to(device)
            missing_flags = batch["missing_flags"].to(device)

            out = model(eeg, sef_sr, missing_flags)
            all_pred_t0.extend(out["bis_t0"].cpu().numpy().flatten().tolist())
            all_pred_t30.extend(out["bis_t30"].cpu().numpy().flatten().tolist())
            all_pred_t60.extend(out["bis_t60"].cpu().numpy().flatten().tolist())
            all_pred_t180.extend(out["bis_t180"].cpu().numpy().flatten().tolist())
            all_pred_t300.extend(out["bis_t300"].cpu().numpy().flatten().tolist())
            all_true_t0.extend(batch["bis_t0"].numpy().flatten().tolist())
            all_true_t30.extend(batch["bis_t30"].numpy().flatten().tolist())
            all_true_t60.extend(batch["bis_t60"].numpy().flatten().tolist())
            all_true_t180.extend(batch["bis_t180"].numpy().flatten().tolist())
            all_true_t300.extend(batch["bis_t300"].numpy().flatten().tolist())
            cids = batch.get("caseid", [0] * len(out["bis_t0"]))
            all_caseids.extend(cids.tolist() if isinstance(cids, torch.Tensor) else list(cids))

            if len(all_pred_t0) >= n_cases * 60:
                break

    unique_cases = list(dict.fromkeys(all_caseids))[:n_cases]
    fig, axes = plt.subplots(n_cases, 1, figsize=(12, 3 * n_cases), squeeze=False)

    for ax_row, cid in zip(axes, unique_cases):
        ax   = ax_row[0]
        idxs = [i for i, c in enumerate(all_caseids) if c == cid]
        t    = list(range(len(idxs)))

        ax.plot(t, [all_true_t0[i]   for i in idxs], color="black",      lw=1.5,  label="True t")
        ax.plot(t, [all_pred_t0[i]   for i in idxs], color="black",      ls="--", label="Pred t",     alpha=0.6)
        ax.plot(t, [all_true_t30[i]  for i in idxs], color="steelblue",  lw=1.5,  label="True t+30s")
        ax.plot(t, [all_pred_t30[i]  for i in idxs], color="steelblue",  ls="--", label="Pred t+30s", alpha=0.6)
        ax.plot(t, [all_true_t60[i]  for i in idxs], color="green",      lw=1.5,  label="True t+60s")
        ax.plot(t, [all_pred_t60[i]  for i in idxs], color="green",      ls="--", label="Pred t+60s", alpha=0.6)
        ax.plot(t, [all_true_t180[i] for i in idxs], color="orange",     lw=1.5,  label="True t+180s")
        ax.plot(t, [all_pred_t180[i] for i in idxs], color="orange",     ls="--", label="Pred t+180s",alpha=0.6)
        ax.plot(t, [all_true_t300[i] for i in idxs], color="coral",      lw=1.5,  label="True t+300s")
        ax.plot(t, [all_pred_t300[i] for i in idxs], color="coral",      ls="--", label="Pred t+300s",alpha=0.6)
        ax.axhline(60, color="red", lw=0.8, ls=":", alpha=0.6, label="BIS=60 (unconscious)")
        ax.set_ylim(0, 100)
        ax.set_ylabel("BIS")
        ax.set_title(f"Case {cid}")
        ax.legend(loc="upper right", fontsize=6, ncol=5)

    axes[-1][0].set_xlabel("Window index")
    plt.suptitle("BIS Model — Predictions vs Ground Truth (t, t+30s, t+60s, t+180s, t+300s)", fontsize=12)
    plt.tight_layout()
    plt.savefig(config["pred_output"], dpi=150)
    plt.close()
    log.info(f"Prediction plot saved → {config['pred_output']}")

In [19]:
# ─────────────────────────────────────────────────────────────────────────────
# 19. Main entry point
# ─────────────────────────────────────────────────────────────────────────────
def main() -> None:
    """End-to-end pipeline: load → preprocess → split → train → evaluate."""
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    log.info(f"Using device: {device}")

    # ── Determine BIS-monitor cases ───────────────────────────────────────────
    try:
        # VitalDB provides a curated list of BIS cases
        bis_cases = vitaldb.find_cases([
                "BIS/BIS",
                "BIS/EEG1_WAV",
                "BIS/EEG2_WAV",
                "BIS/SEF",       # optional but prefer cases that have it
                "BIS/SR",        # optional but prefer cases that have it
            ])
        log.info(f"{len(bis_cases)} cases with BIS/BIS track found")
    except Exception:
        log.warning("vitaldb.find_cases failed — using range 1..6388")
        bis_cases = list(range(1, 6389))

    valid_caseids = []
    bis_cases = sorted(bis_cases)
    for i in bis_cases:
        if i in range(1, 3001):
            valid_caseids.append(i)

    # ── 4. Load data ──────────────────────────────────────────────────────────
    wav_data, scalar_data = load_data(valid_caseids, CONFIG)

    if not wav_data:
        log.error("No data loaded — aborting")
        return

    # ── 8. Patient-level split BEFORE window generation ───────────────────────
    all_loaded = sorted(set(wav_data.keys()) & set(scalar_data.keys()))
    train_ids, val_ids, test_ids = split_cases(
        all_loaded, val_frac=CONFIG["val_frac"],
        test_frac=CONFIG["test_frac"], seed=CONFIG["seed"],
    )

    def subset(d, ids):
        return {c: d[c] for c in ids if c in d}

    # ── 6. Window generation per split ────────────────────────────────────────
    log.info("Generating training windows …")
    train_windows = generate_windows(subset(wav_data, train_ids), subset(scalar_data, train_ids), CONFIG)
    gc.collect()

    log.info("Generating validation windows …")
    val_windows = generate_windows(subset(wav_data, val_ids), subset(scalar_data, val_ids), CONFIG)
    gc.collect()

    log.info("Generating test windows …")
    test_windows = generate_windows(subset(wav_data, test_ids), subset(scalar_data, test_ids), CONFIG)
    del wav_data, scalar_data; gc.collect()

    if not train_windows:
        log.error("No training windows generated — aborting")
        return

    # ── 7. Fit scalers on training data only ─────────────────────────────────
    eeg_mean, eeg_std, sef_sr_mean, sef_sr_std = fit_scalers(train_windows)
    train_windows = apply_scalers(train_windows, eeg_mean, eeg_std, sef_sr_mean, sef_sr_std)
    val_windows   = apply_scalers(val_windows,   eeg_mean, eeg_std, sef_sr_mean, sef_sr_std)
    test_windows  = apply_scalers(test_windows,  eeg_mean, eeg_std, sef_sr_mean, sef_sr_std)

    # ── 9–10. Datasets and DataLoaders ────────────────────────────────────────
    train_ds = BISDataset(train_windows)
    val_ds   = BISDataset(val_windows)
    test_ds  = BISDataset(test_windows)
    train_loader, val_loader, test_loader = build_dataloaders(train_ds, val_ds, test_ds, CONFIG)

    # ── 11. Model ─────────────────────────────────────────────────────────────
    model    = BISModel(CONFIG)
    n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    log.info(f"BISModel parameters: {n_params:,}")

    # ── 16. Train ─────────────────────────────────────────────────────────────
    model = train_model(model, train_loader, val_loader, CONFIG, device)

    # ── 17. Test ──────────────────────────────────────────────────────────────
    evaluate_test(model, test_loader, CONFIG, device)

    # ── 18. Plot ──────────────────────────────────────────────────────────────
    plot_predictions(model, test_loader, CONFIG, device, n_cases=2)

    log.info("BIS model pipeline complete.")




In [ ]:
if __name__ == "__main__":
    main()

2026-06-14 13:01:08 | INFO     | model_bis | Using device: cuda
/tmp/ipykernel_9220/814065170.py:12: DeprecationWarning: find_cases() relies on the per-track 'trks' index, which is deprecated and may be removed in a future release.
  bis_cases = vitaldb.find_cases([
2026-06-14 13:01:09 | INFO     | model_bis | 5569 cases with BIS/BIS track found
Loading cases:  10%|▉         | 253/2623 [04:55<1:01:46,  1.56s/it]